### Canny Edge Detector

In [ ]:
%matplotlib inline
from utils import utils
import canny_edge_detector as ced

In [ ]:
imgs = utils.load_data()
utils.visualize(imgs, 'gray')

In [ ]:
detector = ced.cannyEdgeDetector(imgs, sigma=1.4, kernel_size=5, lowthreshold=0.09, highthreshold=0.17, weak_pixel=100)

In [ ]:
imgs_final = detector.detect()

In [ ]:
utils.visualize(imgs_final, 'gray')

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import cv2
from scipy.ndimage import convolve
from tqdm import tqdm
from PIL import Image

# Utility functions
def load_data(dir_name):
    valid_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tiff")
    imgs = []
    filenames = []
    for filename in os.listdir(dir_name):
        if filename.endswith(valid_extensions):
            img = mpimg.imread(os.path.join(dir_name, filename))
            imgs.append(img)
            filenames.append(filename)
    return imgs, filenames

def visualize(images, cmap=None):
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(20, 5))
    for i in range(n):
        if cmap:
            axes[i].imshow(images[i], cmap=cmap)
        else:
            axes[i].imshow(images[i])
        axes[i].axis('off')
    plt.show()

# Canny edge detector functions
def gaussian_kernel(size, sigma=1.4):
    size = int(size) // 2
    x, y = np.mgrid[-size:size+1, -size:size+1]
    normal = 1 / (2.0 * np.pi * sigma**2)
    g = np.exp(-((x**2 + y**2) / (2.0*sigma**2))) * normal
    return g

def sobel_filters(img):
    Kx = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], np.float32)
    Ky = np.array([[1, 2, 1], [0, 0, 0], [-1, -2, -1]], np.float32)
    
    Ix = convolve(img, Kx)
    Iy = convolve(img, Ky)
    
    G = np.hypot(Ix, Iy)
    G = G / G.max() * 255
    theta = np.arctan2(Iy, Ix)
    
    return G, theta

def non_max_suppression(img, D):
    M, N = img.shape
    Z = np.zeros((M, N), dtype=np.int32)
    angle = D * 180. / np.pi
    angle[angle < 0] += 180

    for i in range(1, M-1):
        for j in range(1, N-1):
            try:
                q = 255
                r = 255
                if (0 <= angle[i, j] < 22.5) or (157.5 <= angle[i, j] <= 180):
                    q = img[i, j+1]
                    r = img[i, j-1]
                elif (22.5 <= angle[i, j] < 67.5):
                    q = img[i+1, j-1]
                    r = img[i-1, j+1]
                elif (67.5 <= angle[i, j] < 112.5):
                    q = img[i+1, j]
                    r = img[i-1, j]
                elif (112.5 <= angle[i, j] < 157.5):
                    q = img[i-1, j-1]
                    r = img[i+1, j+1]

                if (img[i, j] >= q) and (img[i, j] >= r):
                    Z[i, j] = img[i, j]
                else:
                    Z[i, j] = 0

            except IndexError as e:
                pass
    
    return Z

def threshold(img, lowThresholdRatio=0.05, highThresholdRatio=0.09):
    highThreshold = img.max() * highThresholdRatio
    lowThreshold = highThreshold * lowThresholdRatio
    
    M, N = img.shape
    res = np.zeros((M, N), dtype=np.int32)
    
    weak = np.int32(25)
    strong = np.int32(255)
    
    strong_i, strong_j = np.where(img >= highThreshold)
    weak_i, weak_j = np.where((img <= highThreshold) & (img >= lowThreshold))
    
    res[strong_i, strong_j] = strong
    res[weak_i, weak_j] = weak
    
    return res, weak, strong

def hysteresis(img, weak, strong=255):
    M, N = img.shape  
    
    for i in range(1, M-1):
        for j in range(1, N-1):
            if (img[i, j] == weak):
                try:
                    if ((img[i+1, j-1] == strong) or (img[i+1, j] == strong) or (img[i+1, j+1] == strong)
                        or (img[i, j-1] == strong) or (img[i, j+1] == strong)
                        or (img[i-1, j-1] == strong) or (img[i-1, j] == strong) or (img[i-1, j+1] == strong)):
                        img[i, j] = strong
                    else:
                        img[i, j] = 0
                except IndexError as e:
                    pass
    
    return img

def Canny_detector(img, sigma=1.4, kernel_size=5, lowthreshold=0.09, highthreshold=0.17):
    if len(img.shape) == 3:
        img = np.dot(img[..., :3], [0.299, 0.587, 0.114])
    
    img_filtered = convolve(img, gaussian_kernel(kernel_size, sigma=sigma))
    grad, theta = sobel_filters(img_filtered)
    img_nms = non_max_suppression(grad, theta)
    img_thresh, weak, strong = threshold(img_nms, lowThresholdRatio=lowthreshold, highThresholdRatio=highthreshold)
    img_final = hysteresis(img_thresh, weak, strong=strong)
   
    return img_final

def save_images(images, filenames, output_dir, cmap='gray'):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    for img, filename in zip(images, filenames):
        plt.imsave(os.path.join(output_dir, filename), img, cmap=cmap)

def generate_motion_vectors(mv):
    hsv = np.zeros((mv.shape[0], mv.shape[1], 3), dtype='float32')
    hsv[..., 1] = 0.5
    mag = np.sqrt(mv[..., 0]**2 + mv[..., 1]**2)
    ang = np.arctan2(mv[..., 1], mv[..., 0])
    hsv[..., 0] = np.mod(ang / (2 * np.pi) * 255, 255)
    hsv[..., 2] = 0 + 1 * mag / np.max(mag)
    rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)
    return rgb

def detect_motion_vector_edges(mv):
    return Sobel3(mv)

# Implement the Sobel3 function (you should replace this with your actual implementation)
def Sobel3(img):
    Kx = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], np.float32)
    Ky = np.array([[1, 2, 1], [0, 0, 0], [-1, -2, -1]], np.float32)
    Ix = convolve(img, Kx)
    Iy = convolve(img, Ky)
    G = np.hypot(Ix, Iy)
    G = G / G.max() * 255
    return G

# Mock implementation of mv_generator
def mv_generator(file_path, shape):
    # Mock motion vectors (replace with actual implementation)
    for _ in range(len(os.listdir(file_path)) - 1):
        mv = np.random.rand(shape[0], shape[1], 2) * 2 - 1  # Random motion vectors
        yield mv

# Load images from a directory
input_dir = 'nf'
output_dir_canny = 'output_directory_canny_edges'
output_dir_motion_vectors = 'output_directory_motion_vectors'
output_dir_motion_vector_edges = 'output_directory_motion_vector_edges'

imgs, filenames = load_data(input_dir)

# Apply Canny edge detector to all images
canny_imgs = [Canny_detector(img) for img in imgs]
save_images(canny_imgs, filenames, output_dir_canny)

# Generate motion vectors and their edges
motion_vectors = [generate_motion_vectors(mv) for mv in mv_generator(input_dir, (imgs[0].shape[0], imgs[0].shape[1]))]
save_images(motion_vectors, filenames, output_dir_motion_vectors)

motion_vector_edges = [detect_motion_vector_edges(mv) for mv in motion_vectors]
save_images(motion_vector_edges, filenames, output_dir_motion_vector_edges)

# Visualize the results (optional)
visualize(canny_imgs, cmap='gray')
visualize(motion_vectors, cmap=None)
visualize(motion_vector_edges, cmap='gray')